In [8]:
import os
import glob
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
FOLDER_PATH = './sensitivity_model'

# Using a single dictionary for specific output paths
OUTPUT_FILE = {
    'csv': 'ablation_summary.csv',
    'tex': 'table_results.tex'
}

PENALTY_MAPPING = {
    'Weight': 'Weight', 'L2': 'Weight',
    'Jacob_L1': 'Jacob-L1',
    'Jacob_F': 'Jacob-F',
    'Shapley': 'Shap',
    'Fast_Shap': 'F-Shap', 'F-Shap': 'F-Shap'
}

COLUMN_ORDER = ['Weight', 'Jacob-L1', 'Jacob-F', 'Shap', 'F-Shap']
MODEL_ORDER = ['MLP', 'cMLP', 'LSTM', 'cLSTM']

def escape_tex(text):
    """Escapes underscores for LaTeX compatibility."""
    return str(text).replace('_', r'\_')

def parse_filename(filename):
    """Extracts metadata from filename strings."""
    base = os.path.basename(filename)
    name_no_ext = os.path.splitext(base)[0]
    parts = name_no_ext.split('_')
    
    if len(parts) < 6: 
        return None

    raw_penalty = "_".join(parts[5:])
    return {
        'model': parts[4],
        'penalty': PENALTY_MAPPING.get(raw_penalty, raw_penalty)
    }

def process_files(folder_path):
    """Aggregates and processes result files."""
    results_list = []
    files = glob.glob(os.path.join(folder_path, '*'))
    
    for f in files:
        if os.path.isdir(f) or not (f.endswith('.csv') or f.endswith('.txt')):
            continue
            
        meta = parse_filename(f)
        if not meta: continue
        
        try:
            # Rigorous column selection
            df = pd.read_csv(f, usecols=['AUROC', 'AUPRC'])
            
            entry = meta.copy()
            entry.update({
                'AUROC': df['AUROC'].max(), 
                'AUPRC': df['AUPRC'].max()
            })
            results_list.append(entry)
        except Exception as e:
            print(f"File Error: Skipping {f} - {e}")

    return pd.DataFrame(results_list)

def generate_comparative_latex(df, caption, label):
    """Generates a professionally formatted LaTeX table."""
    # Group and aggregate for statistical reporting
    df_agg = df.groupby(['model', 'penalty'])[['AUROC', 'AUPRC']].agg(['mean', 'std'])
    
    latex = [
        r"\begin{table*}[ht]",
        r"    \centering",
        f"    \\caption{{{caption}}}",
        f"    \\label{{{label}}}",
        r"    \providecommand{\res}[2]{\begin{tabular}{@{}c@{}}#1\\[-3pt]{\scriptsize (#2)}\end{tabular}}",
        r"    \resizebox{0.9\textwidth}{!}{%",
        r"    \begin{tabular}{@{} l " + "c" * len(COLUMN_ORDER) + " | " + "c" * len(COLUMN_ORDER) + " @{}}",
        r"        \toprule",
        r"        \multirow{2}{*}{Model} & \multicolumn{" + str(len(COLUMN_ORDER)) + r"}{c}{AUROC} & \multicolumn{" + str(len(COLUMN_ORDER)) + r"}{c}{AUPRC} \\",
        r"        \cmidrule(lr){2-" + str(len(COLUMN_ORDER)+1) + r"} \cmidrule(lr){" + str(len(COLUMN_ORDER)+2) + r"-" + str(2*len(COLUMN_ORDER)+1) + r"}",
        "         & " + " & ".join(COLUMN_ORDER) + " & " + " & ".join(COLUMN_ORDER) + r" \\",
        r"        \midrule"
    ]

    for model in MODEL_ORDER:
        if model not in df_agg.index.get_level_values('model'):
            continue
        
        row = [f"        {escape_tex(model)}"]
        
        for metric in ['AUROC', 'AUPRC']:
            for pen in COLUMN_ORDER:
                try:
                    mean_val = df_agg.loc[(model, pen), (metric, 'mean')]
                    std_val = df_agg.loc[(model, pen), (metric, 'std')]
                    row.append(f"\\res{{{mean_val:.3f}}}{{{std_val:.3f}}}")
                except KeyError:
                    row.append("--")
        
        latex.append(" & ".join(row) + r" \\")

    latex.extend([
        r"        \bottomrule",
        r"    \end{tabular}%",
        r"    }",
        r"\end{table*}"
    ])
    
    return "\n".join(latex)

if __name__ == "__main__":
    df_results = process_files(FOLDER_PATH)

    if not df_results.empty:
        # Save results to the specified CSV
        # df_results.to_csv(OUTPUT_FILE['csv'], index=False)
        
        # Generate and save the TeX table
        tex_output = generate_comparative_latex(
            df_results, 
            caption="Comparison of performance metrics across penalty configurations.",
            label="tab:performance_summary"
        )
        
        # with open(OUTPUT_FILE['tex'], 'w') as f:
        #     f.write(tex_output)
        
        print(tex_output)
        print(f"Success: Processed {len(df_results)} records.")
        print(f"Outputs: {OUTPUT_FILE['csv']}, {OUTPUT_FILE['tex']}")
    else:
        print("Error: No data found. Check your FOLDER_PATH and filename patterns.")

\begin{table*}[ht]
    \centering
    \caption{Comparison of performance metrics across penalty configurations.}
    \label{tab:performance_summary}
    \providecommand{\res}[2]{\begin{tabular}{@{}c@{}}#1\\[-3pt]{\scriptsize (#2)}\end{tabular}}
    \resizebox{0.9\textwidth}{!}{%
    \begin{tabular}{@{} l ccccc | ccccc @{}}
        \toprule
        \multirow{2}{*}{Model} & \multicolumn{5}{c}{AUROC} & \multicolumn{5}{c}{AUPRC} \\
        \cmidrule(lr){2-6} \cmidrule(lr){7-11}
         & Weight & Jacob-L1 & Jacob-F & Shap & F-Shap & Weight & Jacob-L1 & Jacob-F & Shap & F-Shap \\
        \midrule
        MLP & -- & \res{0.859}{0.026} & \res{0.844}{0.030} & \res{0.874}{0.015} & \res{0.855}{0.025} & -- & \res{0.645}{0.038} & \res{0.640}{0.036} & \res{0.673}{0.021} & \res{0.659}{0.029} \\
        cMLP & -- & \res{0.827}{0.023} & \res{0.831}{0.018} & \res{0.854}{0.012} & \res{0.844}{0.018} & -- & \res{0.630}{0.020} & \res{0.633}{0.015} & \res{0.660}{0.014} & \res{0.639}{0.021} \\
        LSTM 